# 第 10 章实践：调优与选做挑战


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）online softmax 三运行量 ______、______、______；修正系数 ______。
2. （选择）核内 K 块循环的同步归属：
   A. auto_mutex 管 Tile 流水即可<br/>
   B. 需要 Cube/Vector 显式跨核同步（或自动流水）
3. （判断）实现完全遮蔽块跳过后，各 Q 块的 causal 任务量是均匀的。（对/错）

In [ ]:
!cat ./answer/10.06_objective_answer.txt

## 2. 编程实践一（简单）：编写分块规划函数

**任务**：编写独立的 Host 函数 `plan_rows(seq, block=256)`，返回各块的 `(起点, 有效行数)`；检查覆盖完整、无重叠、无越界，用断言验证 S=2048 与 S=3000。该函数只检查分块算术，不模拟 Device 执行。随后保持算法不变，完成两组推演——① 把 H 从 64 改为 128：按 10.03 预算表重算各 Tile 字节与 UB 占用，判断原地址布局是否仍安全；② 把 S 从 2048 改为 3000（q_block=kv_block=256 的扩展尾块用例，不替代 RFC 四档 S）：指出需要尾块处理的 Q 块编号与 valid 形状。完成这两组推演后，在 950 上实际改参重跑，检查结果是否符合推演。


In [ ]:
# 在此实现 plan_rows，并为 2048、3000 编写覆盖与尾块断言
# 再记录 H=128 的预算推演：逻辑字节、每个 AIV 的物理地址与缓冲槽数
pass

## 3. 编程实践二（中等）：跑通教学版并建立基线

**任务**：使用 10.04 的完整串行参考，在 950 跑 S=2048、H=64、causal/无 mask 两组并完成精度验收。通过 `prepare_attention` 预分配，预热其返回的启动闭包，再按第 8 章方法计时；分别记录 workspace、Host 墙钟与 profiler 设备时间。随后比较片上中转或多槽版本，明确每项改变删除了哪些搬运或等待。


In [ ]:
# 在此完成实践二（运行 + 验收 + 基线记录）
pass

## 4. 编程实践三（困难）：性能调优闭环

**任务**：按第 8 章"四问"分析基线，实施至少两项优化（候选：K/V 双缓冲加深、块级流水化、kv_block 尺寸扫描、Phase 启用），每项**先精度回归再测性能**，填表：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">版本</th><th style="text-align:left;">S=2048 耗时</th><th style="text-align:left;">S=4096 耗时</th><th style="text-align:left;">备注</th></tr>
<tr><td>教学版基线</td><td>【950 待测】</td><td>【950 待测】</td><td></td></tr>
<tr><td>优化 1：______</td><td>【950 待测】</td><td>【950 待测】</td><td></td></tr>
<tr><td>优化 2：______</td><td>【950 待测】</td><td>【950 待测】</td><td></td></tr>
</table>


In [ ]:
!cat ./answer/10.06_practice_answer.txt

### 完整参考实现与验收

参考已补齐 QK/online/PV/归一化及 Host 验收；流水、片上中转和 Phase 是基于该基线的优化任务，尚无设备性能结论。H=64/128、动态 B/S、原始位置 causal 与零有效行均有明确处理。

[参考源码](./src/sdpa_reference.py) 可在 950 上从章节目录执行 `python src/sdpa_reference.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/sdpa_reference.py


## 5. 本节小结

SDPA 实战集齐了课程的全部武器：规格推导、Tiling 设计、CV 协同、数值纪律、调优闭环。第 11 章在 SDPA 之上加"门控 KV 选择"——把第 9 章的 Select-then-Compute 与本章的 flash 循环合体，为阅读 gym 中的稀疏 MLA/Indexer 算子建立基础；教学模型不等同于特定模型的完整算法。